In [20]:
import torch
from torch import nn
from torch.nn import functional as F
from kerops.ops.linear.relu_linear_backward import ReLULinearBackward, autotune_relu_lin_backward, generate_inputs_relu_lin_backward
from kerops.ops.assets import ASSETS_ROOT

In [2]:
autotune_relu_lin_backward(ASSETS_ROOT / 'ReLULinBackward.toml', n_jobs_precompile=4)

Problem sizes:   0%|          | 0/3 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/72 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/72 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/72 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/72 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/72 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/72 [00:00<?, ?it/s]

In [32]:
channels = 128

x, grad, weight = generate_inputs_relu_lin_backward({'in_channels': channels})

In [33]:
%%timeit -r 10 -n 10
ReLULinearBackward(x, grad, weight)
torch.cuda.synchronize()

423 μs ± 30.4 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [34]:
model = nn.Sequential(
    nn.ReLU(),
    nn.Conv3d(x.shape[1], grad.shape[1], kernel_size=1, bias=False)
).half().cuda()

with torch.amp.autocast('cuda'):
    out = model(x)

In [35]:
%%timeit -r 10 -n 10
out.backward(grad, retain_graph=True)
torch.cuda.synchronize()

241 μs ± 58.2 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)
